# FTM NIPS 2017 Surrogate Model Experiment

This notebook documents the experiment run on the NIPS 2017 ImageNet-compatible dataset using the `ftm_diff_model` implementation.

## Objective

The goal was to evaluate the effect of changing the surrogate model used to generate targeted adversarial examples with FTM, while keeping the attack setup, dataset, and target-model evaluation fixed.

## Dataset

- Dataset: NIPS 2017 ImageNet-compatible dataset
- Number of images: 1000
- Image metadata: `data/images.csv`
- Image folder: `data/images/`
- Each image has a true ImageNet label and a target ImageNet label for targeted attack evaluation.

In [ ]:
import csv, glob, os

image_files = glob.glob('data/images/*.png')
with open('data/images.csv', newline='') as f:
    rows = list(csv.DictReader(f))

image_ids = {os.path.splitext(os.path.basename(p))[0] for p in image_files}
missing = [r['ImageId'] for r in rows if r['ImageId'] not in image_ids]

print('Images:', len(image_files))
print('CSV rows:', len(rows))
print('Missing:', len(missing))

## Attack Setup

- Attack: RDI-TI-MI-FTM (`RTMF` in code)
- Iterations: 300
- Epsilon: 16/255
- Step size: 2/255
- Targeted attack: True
- Evaluation: Attack Success Rate (ASR) against 15 target models

The core FTM attack logic in `attacks.py` was not changed. The modified code mainly adds CPU/GPU compatibility and result logging.

## Surrogate Models

Three surrogate/source models were tested:

1. ResNet50: baseline ImageNet-supervised surrogate used in the original FTM setup
2. ConvNeXt Base: modern CNN architecture trained on ImageNet
3. CLIP RN50: ResNet-style CLIP visual backbone with image-text contrastive pretraining

## Commands Used

In [ ]:
# ResNet50 surrogate
!python main.py --device cuda:0 --batch_size 20 --model_name ResNet50 --save_dir ./exp/ResNet50/ftm_nips --eval

# ConvNeXt surrogate
!python main.py --device cuda:0 --batch_size 20 --model_name ConvNeXt --save_dir ./exp/ConvNeXt/ftm_nips --eval

# CLIP RN50 surrogate
# Requires OpenAI CLIP package:
# !pip install -q ftfy regex tqdm
# !pip install -q git+https://github.com/openai/CLIP.git
!python main.py --device cuda:0 --batch_size 20 --model_name CLIP_RN50 --save_dir ./exp/CLIP_RN50/ftm_nips --eval

## Model-Wise Results

Attack Success Rate (ASR %) on 1000 NIPS images.

In [ ]:
import pandas as pd

results = pd.DataFrame({
    'Target model': [
        'ResNet18', 'ResNet50', 'vgg16', 'inception_v3', 'efficientnet_b0',
        'DenseNet121', 'mobilenet_v2', 'inception_resnet_v2', 'inception_v4_timm',
        'xception', 'vit_base_patch16_224', 'levit_384', 'convit_base',
        'twins_svt_base', 'pit'
    ],
    'ResNet50': [87.0, 97.7, 85.4, 68.3, 81.5, 89.7, 82.9, 52.6, 65.3, 56.8, 13.2, 52.3, 10.8, 30.9, 31.4],
    'ConvNeXt': [51.2, 57.0, 38.5, 37.6, 65.0, 60.5, 44.7, 34.8, 32.8, 34.8, 46.8, 79.1, 46.5, 69.9, 64.8],
    'CLIP RN50': [5.4, 6.0, 9.4, 5.1, 7.4, 7.3, 4.7, 3.6, 5.8, 5.3, 0.1, 3.1, 0.3, 0.7, 0.9]
})

results

In [ ]:
overall = pd.DataFrame({
    'Surrogate': ['ResNet50', 'ConvNeXt', 'CLIP RN50'],
    'Overall ASR (%)': [62.39, 53.60, 4.35]
})

overall

In [ ]:
cnn_targets = [
    'ResNet18', 'ResNet50', 'vgg16', 'inception_v3', 'efficientnet_b0',
    'DenseNet121', 'mobilenet_v2', 'inception_resnet_v2', 'inception_v4_timm', 'xception'
]
transformer_targets = ['vit_base_patch16_224', 'levit_384', 'convit_base', 'twins_svt_base', 'pit']

family_summary = pd.DataFrame({
    'Surrogate': ['ResNet50', 'ConvNeXt', 'CLIP RN50'],
    'CNN target avg ASR (%)': [
        results[results['Target model'].isin(cnn_targets)]['ResNet50'].mean(),
        results[results['Target model'].isin(cnn_targets)]['ConvNeXt'].mean(),
        results[results['Target model'].isin(cnn_targets)]['CLIP RN50'].mean(),
    ],
    'Transformer target avg ASR (%)': [
        results[results['Target model'].isin(transformer_targets)]['ResNet50'].mean(),
        results[results['Target model'].isin(transformer_targets)]['ConvNeXt'].mean(),
        results[results['Target model'].isin(transformer_targets)]['CLIP RN50'].mean(),
    ],
})

family_summary.round(2)

## Conclusion

The surrogate model has a clear effect on targeted transferability.

- ResNet50 gives the highest overall ASR and transfers especially well to CNN-based target models.
- ConvNeXt gives lower overall ASR than ResNet50, but transfers much better to transformer-oriented target models.
- CLIP RN50 performs poorly overall, suggesting that a similar ResNet-style backbone is not enough for strong transfer when the training objective and learned representation are different.

Therefore, both surrogate architecture and learned representation/training objective influence transferability.